# Introduction to Data Science 2026

# Week 4

In this week's exercise, we look at prompting and zero- and few-shot task settings. Below is a text generation example from https://github.com/TurkuNLP/intro-to-nlp/blob/master/text_generation_pipeline_example.ipynb demonstrating how to load a text generation pipeline with a pre-trained model and generate text with a given prompt. Your task is to load a similar pre-trained generative model and assess whether the model succeeds at a set of tasks in zero-shot, one-shot, and two-shot settings.

**Note: Downloading and running the pre-trained model locally may take some time. Alternatively, you can open and run this notebook on [Google Colab](https://colab.research.google.com/), as assumed in the following example.**

## Text generation example

This is a brief example of how to run text generation with a causal language model and `pipeline`.

Install [transformers](https://huggingface.co/docs/transformers/index) python package. This will be used to load the model and tokenizer and to run generation.

In [1]:
!pip install --quiet transformers

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load a generative model and its tokenizer. You can substitute any other generative model name here (e.g. [other TurkuNLP GPT-3 models](https://huggingface.co/models?sort=downloads&search=turkunlp%2Fgpt3)), but note that Colab may have issues running larger models. 

In [4]:
MODEL_NAME = 'TurkuNLP/gpt3-finnish-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 293/293 [00:00<00:00, 35269.52it/s]


Instantiate a text generation pipeline using the tokenizer and model.

In [5]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [6]:
output = pipe('Terve, miten menee?', max_new_tokens=25)

print(output)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Terve, miten menee?”\n”Ihan hyvin.”\n”Haluatko, että tulen mukaan?”\n”Ei kiitos.”\n”Minne?”'}]


Just print the text

In [7]:
print(output[0]['generated_text'])

Terve, miten menee?”
”Ihan hyvin.”
”Haluatko, että tulen mukaan?”
”Ei kiitos.”
”Minne?”


We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:

In [8]:
output = pipe(
    'Terve, miten menee?',
    do_sample=True,
    temperature=10.0,
    max_new_tokens=25
)

print(output[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=25) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Terve, miten menee? Oletko kunnossa vielä. Me oltiin yötä kaverilla, sillä he rakensivat talon tai sellaisen hirsien päättelivät jo tänään niin... Nyt aamulla


## Exercise 1

Your task is to assess whether a generative model succeeds in the following tasks in zero-shot, one-shot, and two-shot settings:

- binary sentiment classification (positive / negative)

- person name recognition

- two-digit addition (e.g. 11 + 22 = 33)

For example, for assessing whether a generative model can name capital cities, we could use the following prompts:

- zero-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- one-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- two-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Denmark?\
	>Answer: Copenhagen
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""

You can do the tasks either in English or Finnish and use a generative model of your choice from the Hugging Face models repository, for example the following models:

- English: `gpt2-large`
- Finnish: `TurkuNLP/gpt3-finnish-large`

You can either come up with your own instructions for the tasks or use the following:

- English:
	- binary sentiment classification: "Do the following texts express a positive or negative sentiment?"
	- person name recognition: "List the person names occurring in the following texts."
	- two-digit addition: "This is a first grade math exam."
- Finnish:
	- binary sentiment classification: "Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?"
	- person name recognition: "Listaa seuraavissa teksteissä mainitut henkilönnimet."
	- two-digit addition: "Tämä on ensimmäisen luokan matematiikan koe."

Come up with at least two test cases for each of the three tasks, and come up with your own one- and two-shot examples.

In [9]:
ENG_MODEL_NAME = 'gpt2-large'
eng_tokenizer = AutoTokenizer.from_pretrained(ENG_MODEL_NAME)
eng_model = AutoModelForCausalLM.from_pretrained(ENG_MODEL_NAME)
pipe_eng = pipeline(
    'text-generation',
    model=eng_model,
    tokenizer=eng_tokenizer,
    device=eng_model.device
)
sentiment_prompts = {
    "zero-shot": [
        """Do the following texts express a positive or negative sentiment?

Text: I absolutely loved this restaurant. The food was delicious and the service was excellent.
Answer:""",

        """Do the following texts express a positive or negative sentiment?

Text: The movie was boring and I regretted spending money on it.
Answer:"""
    ],

    "one-shot": [
        """Do the following texts express a positive or negative sentiment?

Text: The hotel was wonderful and the staff were very friendly.
Answer: positive

Text: I absolutely loved this restaurant. The food was delicious and the service was excellent.
Answer:""",

        """Do the following texts express a positive or negative sentiment?

Text: The hotel was wonderful and the staff were very friendly.
Answer: positive

Text: The movie was boring and I regretted spending money on it.
Answer:"""
    ],

    "two-shot": [
        """Do the following texts express a positive or negative sentiment?

Text: The hotel was wonderful and the staff were very friendly.
Answer: positive

Text: The flight was delayed for five hours and the staff were unhelpful.
Answer: negative

Text: I absolutely loved this restaurant. The food was delicious and the service was excellent.
Answer:""",

        """Do the following texts express a positive or negative sentiment?

Text: The hotel was wonderful and the staff were very friendly.
Answer: positive

Text: The flight was delayed for five hours and the staff were unhelpful.
Answer: negative

Text: The movie was boring and I regretted spending money on it.
Answer:"""
    ]
}
name_prompts = {
    "zero-shot": [
        """List the person names occurring in the following text.

Text: Sarah met James at the train station.
Answer:""",

        """List the person names occurring in the following text.

Text: Michael called Emma after speaking with David.
Answer:"""
    ],

    "one-shot": [
        """List the person names occurring in the following text.

Text: John went to the library with Alice.
Answer: John, Alice

Text: Sarah met James at the train station.
Answer:""",

        """List the person names occurring in the following text.

Text: John went to the library with Alice.
Answer: John, Alice

Text: Michael called Emma after speaking with David.
Answer:"""
    ],

    "two-shot": [
        """List the person names occurring in the following text.

Text: John went to the library with Alice.
Answer: John, Alice

Text: Peter had lunch with Maria.
Answer: Peter, Maria

Text: Sarah met James at the train station.
Answer:""",

        """List the person names occurring in the following text.

Text: John went to the library with Alice.
Answer: John, Alice

Text: Peter had lunch with Maria.
Answer: Peter, Maria

Text: Michael called Emma after speaking with David.
Answer:"""
    ]
}
addition_prompts = {
    "zero-shot": [
        """This is a first grade math exam.

Question: 24 + 35 =
Answer:""",

        """This is a first grade math exam.

Question: 47 + 21 =
Answer:"""
    ],

    "one-shot": [
        """This is a first grade math exam.

Question: 12 + 23 =
Answer: 35

Question: 24 + 35 =
Answer:""",

        """This is a first grade math exam.

Question: 12 + 23 =
Answer: 35

Question: 47 + 21 =
Answer:"""
    ],

    "two-shot": [
        """This is a first grade math exam.

Question: 12 + 23 =
Answer: 35

Question: 31 + 14 =
Answer: 45

Question: 24 + 35 =
Answer:""",

        """This is a first grade math exam.

Question: 12 + 23 =
Answer: 35

Question: 31 + 14 =
Answer: 45

Question: 47 + 21 =
Answer:"""
    ]
}
def run_tests(name, prompts):
    print("=" * 60)
    print(name)
    print("=" * 60)

    for setting, test_prompts in prompts.items():
        print(f"\n--- {setting} ---")

        for i, prompt in enumerate(test_prompts, 1):
            output = pipe_eng(
                prompt,
                max_new_tokens=15,
                do_sample=False
            )

            generated_text = output[0]["generated_text"]
            answer = generated_text[len(prompt):]

            print(f"Test case {i}: {answer.strip()}")


run_tests("SENTIMENT CLASSIFICATION", sentiment_prompts)
run_tests("PERSON NAME RECOGNITION", name_prompts)
run_tests("TWO-DIGIT ADDITION", addition_prompts)

Loading weights: 100%|██████████| 436/436 [00:03<00:00, 109.05it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SENTIMENT CLASSIFICATION

--- zero-shot ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test case 1: I absolutely loved this restaurant. The food was delicious and the service was excellent


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Test case 1: 36

Question: 25 + 36 =

Answer: 37


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test case 2: 42

Question: 42 + 21 =

Answer: 42

--- two-shot ---


[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test case 1: 55

Question: 31 + 14 + 35 =

Answer:
Test case 2: 55

Question: 56 + 23 =

Answer: 61


**Submit this exercise by submitting your code and your answers to the above questions as comments on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**